# Session 6 · Block 2 – Kafka from Python: producers, consumers, consumer groups  (≈ 60 min)

**Runs locally** with your `.venv` kernel and the broker from Block 1 (Docker or native) on `localhost:9092`.
Open this notebook from the `session-06/` folder – it imports `kafka_data.py` from there.

What you learn:
1. create a topic with 3 partitions from Python – *idempotently* (safe to run twice),
2. what a **serializer** does,
3. **keyed vs. unkeyed** messages: which partition, which order,
4. consumers, **offsets**, auto-commit, lag and *at-least-once*,
5. consumer **groups** with real, independent processes (terminal scripts `producer.py` / `consumer.py`).

All loops in this notebook are **bounded** (fixed number of messages, consumer stops after a few idle seconds),
so no cell runs forever. Endless producers and consumers run in the terminal instead (part 7).

| Library | Why |
|---|---|
| `kafka-python` ≥ 3.0 | pure-Python Kafka client (version 2.0.2 from older tutorials does **not** work on Python 3.12) |
| `faker` | fake names for the events |

## 0. Is the broker there?
If this cell fails with `Unable to bootstrap` / `KafkaTimeoutError`: the broker is not running → README.md, *Check the broker*.

In [ ]:
import json
from collections import Counter, defaultdict

from kafka import (DefaultSerializer, JsonSerializer, KafkaAdminClient, KafkaConsumer,
                   KafkaProducer, TopicPartition)
from kafka.admin import NewTopic
from kafka.errors import KafkaError, TopicAlreadyExistsError

from kafka_data import KeyedEvents

BOOTSTRAP = "localhost:9092"
TOPIC = "user-events"          # the same topic the terminal scripts use

try:
    admin = KafkaAdminClient(bootstrap_servers=BOOTSTRAP, client_id="notebook-admin")
except KafkaError as e:
    raise RuntimeError(f"No broker at {BOOTSTRAP} – start it first (README.md). Details: {e}") from None
print("✓ broker answers. Topics:", sorted(admin.list_topics()))

## 1. Create the topic – idempotently
*Idempotent* = running it once or ten times leaves the same state. Creating a topic that already exists raises
`TopicAlreadyExistsError` – we catch it, so you can re-run this cell (and the terminal scripts can create the same topic).

Our broker has `auto.create.topics.enable=false`: a typo in a topic name gives an error instead of a new, empty topic.

In [ ]:
try:
    admin.create_topics([NewTopic(name=TOPIC, num_partitions=3, replication_factor=1)])
    print(f"created {TOPIC!r}")
except TopicAlreadyExistsError:
    print(f"{TOPIC!r} already exists – fine")

for t in admin.describe_topics([TOPIC]):
    for p in sorted(t["partitions"], key=lambda p: p["partition_index"]):
        print(f"partition {p['partition_index']}: leader={p['leader_id']} "
              f"replicas={p['replica_nodes']} isr={p['isr_nodes']}")
    if len(t["partitions"]) != 3:
        print("⚠ this topic does not have 3 partitions – delete it (Block 1, part 7) and run the cell again")

## 2. Serializers: from a Python object to bytes
Kafka only stores **bytes** (Block 1, the hex dump). The producer needs a *serializer* (object → bytes),
the consumer a *deserializer* (bytes → object). The old lab wrote it by hand – `kafka-python` 3 also ships ready-made ones.

In [ ]:
value = {"state": "Tirol", "seq": 1, "action": "login"}

by_hand = json.dumps(value).encode("utf-8")                    # what the old lab's json_serializer did
built_in = JsonSerializer().serialize(TOPIC, [], value)        # what we use from now on
print(by_hand)
print("same bytes:", by_hand == built_in, "·", len(by_hand), "bytes")
print("back to a dict:", JsonSerializer().deserialize(TOPIC, [], by_hand))

## 3. A keyed producer
`acks="all"`: the broker answers only when the write is complete on all in-sync replicas.
`.get(timeout=10)` waits for that answer and returns **where** the message went: partition and offset.

Every event has a key (an Austrian state) and a `seq` number that counts **per key**: Tirol 1, 2, 3, …

In [ ]:
producer = KafkaProducer(
    bootstrap_servers=[BOOTSTRAP],
    key_serializer=DefaultSerializer(),    # "Tirol" -> b"Tirol"
    value_serializer=JsonSerializer(),     # dict -> JSON bytes
    acks="all",
    retries=5,
    linger_ms=10,
)

keyed = KeyedEvents()                       # one generator = one "run" id
sent = []                                   # (key, seq, partition, offset)
for _ in range(27):
    key, value = keyed.next()
    meta = producer.send(TOPIC, key=key, value=value).get(timeout=10)
    sent.append((key, value["seq"], meta.partition, meta.offset))

for row in sent[:10]:
    print("key={:<18} seq={:<3} -> partition {} @ offset {}".format(*row))
print("…", len(sent), "messages sent")

### ✏️ Your turn: which partitions did each key use?
Build a dictionary `key -> set of partitions` from `sent`. **Predict first:** how many partitions per key?

In [ ]:
key_to_partitions = defaultdict(set)
# TODO: fill key_to_partitions from `sent`  (each row is: key, seq, partition, offset)

for key in sorted(key_to_partitions):
    print(f"{key:<18} -> partitions {sorted(key_to_partitions[key])}")

Each key appears in exactly **one** partition: `partition = hash(key) mod 3`. Several keys share a partition – that is fine.
The Java CLI from Block 1 and this Python client use the same hash function (murmur2), so `Tirol` lands in the same
partition from both.

## 4. The same events without a key

In [ ]:
unkeyed = KeyedEvents()                     # new run id, but we will NOT use the key
unkeyed_sent = []
for _ in range(27):
    key, value = unkeyed.next()
    meta = producer.send(TOPIC, key=None, value=value).get(timeout=10)
    unkeyed_sent.append((value["state"], value["seq"], meta.partition))

print("messages per partition:", dict(sorted(Counter(p for _, _, p in unkeyed_sent).items())))
per_state = defaultdict(set)
for state, seq, p in unkeyed_sent:
    per_state[state].add(p)
for state in sorted(per_state):
    print(f"{state:<18} -> partitions {sorted(per_state[state])}")

Without a key, the producer spreads the messages – the same state now ends up in **several** partitions.
(How exactly it spreads them depends on the client library; the Java client fills one partition for a while, then switches.)

## 5. Consume – with a limit
`consumer_timeout_ms=5000` ends the `for` loop after 5 s without new messages. A **new** `group_id` has no committed
offsets yet, so `auto_offset_reset="earliest"` makes it start at the beginning of every partition.

In [ ]:
GROUP = f"notebook-{keyed.run}"             # a fresh group name every time the kernel restarts

consumer = KafkaConsumer(
    TOPIC,
    bootstrap_servers=[BOOTSTRAP],
    group_id=GROUP,
    auto_offset_reset="earliest",
    enable_auto_commit=True,                # commit offsets in the background every 5 s (and on close)
    key_deserializer=DefaultSerializer(),
    value_deserializer=JsonSerializer(),
    consumer_timeout_ms=5000,
)
records = [r for r in consumer]             # stops after 5 s of silence
consumer.close()                            # commits the final offsets and leaves the group

print(len(records), "records ·", "per partition:", dict(sorted(Counter(r.partition for r in records).items())))
for r in records[:8]:
    print(f"p{r.partition} @ {r.offset:<4} key={str(r.key):<18} seq={r.value.get('seq')}")

### Is the order kept?
We check the `seq` numbers of **our two runs** in the order the consumer received them:
for every `(run, state)` the seq must go 1, 2, 3, … – otherwise we count an *order violation*.

In [ ]:
def order_violations(recs, run):
    last, bad = {}, 0
    for r in recs:
        v = r.value
        if not isinstance(v, dict) or v.get("run") != run:
            continue
        state = v["state"]
        if state in last and v["seq"] != last[state] + 1:
            bad += 1
        last[state] = v["seq"]
    return bad

print("keyed run   – order violations per key:", order_violations(records, keyed.run))
print("unkeyed run – order violations per key:", order_violations(records, unkeyed.run))

Keyed: **0** violations – one key = one partition, and inside a partition order is guaranteed.
Unkeyed: usually **> 0** – the consumer reads partition by partition, so seq 3 of *Tirol* from partition 0 can arrive
before seq 2 from partition 2. **Kafka guarantees order per partition, not per topic.**
(If you see 0 here, run sections 4 and 5 again – with few messages it can happen by chance.)

## 6. Offsets, commits and lag
Read with the **same group** again. ✏️ Predict: how many records do you get?

In [ ]:
again = KafkaConsumer(TOPIC, bootstrap_servers=[BOOTSTRAP], group_id=GROUP,
                      auto_offset_reset="earliest", consumer_timeout_ms=3000)
print("same group, second read:", sum(1 for _ in again), "records")
again.close()

0 – the group committed its offsets, so it continues where it stopped. Now produce 5 messages while nobody reads,
then compute the **lag** = log-end offset − committed offset, per partition.

In [ ]:
for _ in range(5):
    key, value = keyed.next()
    producer.send(TOPIC, key=key, value=value)
producer.flush()

probe = KafkaConsumer(bootstrap_servers=[BOOTSTRAP], group_id=GROUP, enable_auto_commit=False)
partitions = [TopicPartition(TOPIC, p) for p in sorted(probe.partitions_for_topic(TOPIC))]
end = probe.end_offsets(partitions)               # {TopicPartition: log-end offset}
committed = {tp: probe.committed(tp) or 0 for tp in partitions}
probe.close()
print("log-end:  ", {tp.partition: o for tp, o in end.items()})
print("committed:", {tp.partition: o for tp, o in committed.items()})

### ✏️ Your turn: lag per partition and in total
Use `end` and `committed` from the cell above. The total must be 5 – why?

In [ ]:
lag = {}
# TODO: lag[partition number] = end offset - committed offset

print("lag per partition:", lag, "· total:", sum(lag.values()))

### At-least-once
With `enable_auto_commit=True` the consumer commits **after** it has received messages, every 5 s.
If it crashes *after processing* but *before the next commit*, the restarted consumer gets those messages **again**
→ **at-least-once**: nothing is lost, but duplicates are possible. Your processing should therefore be *idempotent*
(e.g. upsert by an event id instead of `count += 1`).

| Commit … | Crash in between means | Name |
|---|---|---|
| after processing | messages processed twice | at-least-once |
| before processing | messages lost | at-most-once |

Kafka also offers *exactly-once* within Kafka (transactions) – not part of this course.

## 7. Consumer groups with real processes – terminal experiments
Notebook cells run one after another, so we need **separate terminals** for parallel consumers.
Open 3–4 terminals in VS Code (**+** in the terminal panel), activate `.venv` in each, `cd session-06`.

**Experiment 1 – one producer, one consumer**
```bash
python consumer.py --name C1                       # terminal 1
python producer.py --interval 0.5                  # terminal 2
```
`C1` prints `ASSIGNED partitions [0, 1, 2]` and then every message as `p<partition> @ <offset> key=… seq=…`.

**Experiment 2 – a second consumer in the SAME group.** ✏️ Predict first: will C2 get copies of all messages, or a part?
```bash
python consumer.py --name C2                       # terminal 3, same default group: group-a
```
Watch C1: `REVOKED` and a new `ASSIGNED` line = a **rebalance**. Then look at the assignment from the broker's view:

| Route A (Docker) | Route B (native) |
|---|---|
| `docker exec -it broker /opt/kafka/bin/kafka-consumer-groups.sh --bootstrap-server localhost:9092 --describe --group group-a` | `kafka-consumer-groups.sh --bootstrap-server localhost:9092 --describe --group group-a` |

The `CONSUMER-ID` column shows which consumer owns which partition. ✏️ Now start **C3 and C4** in the same group.
With 3 partitions – what does C4 do? Stop one consumer with Ctrl+C: what happens to its partitions?

**Experiment 3 – a DIFFERENT group.** ✏️ Predict: what does `B1` receive?
```bash
python consumer.py --group group-b --name B1       # terminal 4
```
B1 reads **everything from the beginning** and then the same live messages as group-a: every group gets all
messages, *inside* a group each partition goes to exactly one consumer.

**Bonus – keys and order:** stop the producer and run `python producer.py --no-key --interval 0.2`.
Consumers in group-a may now print `⚠ expected seq …` – the same state is spread over partitions read by different consumers.

## 8. Exam-style questions
1. *(2 P)* What does a serializer do in a Kafka producer, and why does Kafka itself not care whether you send JSON, Avro or CSV?
2. *(3 P)* A topic has 3 partitions and one consumer group with 4 consumers. How are the partitions assigned, and what
   does the 4th consumer do? What changes if one consumer crashes?
3. *(3 P)* Two applications – a fraud detector and a dashboard – both need **all** payment events. Same `group_id` or
   different ones? Justify.
4. *(4 P)* Explain *at-least-once* delivery with auto-commit. Describe a concrete situation that creates a duplicate,
   and one way to make the consumer robust against it.
5. *(4 P)* An online shop needs all events of one order processed in the correct order, but also wants to scale to
   many consumers. How do you choose the message key, and why does that solve both requirements?

In [ ]:
producer.close()
admin.close()
print("closed")